In [1]:
import json
import os
import sys
from pathlib import Path

import brainunit as u

os.environ.setdefault("JAX_PLATFORMS", "cpu")


def find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "braincell").exists() and (candidate / "develop_doc").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository root or a subdirectory inside it.")


repo_root = find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import braincell
from braincell import Morpho
from braincell.io import NeuroMorphoClient, find_standard_swc, load_cached_metadata

print("braincell version:", braincell.__version__)
print("repo_root:", repo_root)


/home/swl/anaconda3/envs/braincell/lib/python3.10/site-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.5.1 is installed, but it is not compatible with the installed jaxlib version 0.6.2, so it will not be used.
  warnings.warn(


braincell version: 0.0.8
repo_root: /home/swl/braincell


# `neuromorpho_diff`: batch cache NeuroMorpho neurons, then batch compare with `braincell`

This notebook is organized as two direct steps:

1. use `q` and `fq` to batch query NeuroMorpho neurons
2. cache every neuron under `develop_doc/data/neuromorpho/<neuron_id>/`
3. store `metadata.json` with neuron metadata, measurement, and download links
4. scan cached folders, load standardized `SWC`, and compare against NeuroMorpho measurement


In [2]:
EXACT_MATCH_KEYS = ("n_stems", "n_branch", "n_bifs")
FLOAT_THRESHOLDS = {
    "length": 0.001,
    "surface": 0.01,
    "volume": 0.01,
    "pathDistance": 0.001,
    "eucDistance": 0.001,
    "soma_Surface": 0.10,
}
REQUIRED_MEASUREMENT_KEYS = EXACT_MATCH_KEYS + tuple(FLOAT_THRESHOLDS)


def missing_measurement_keys(measurement):
    return [key for key in REQUIRED_MEASUREMENT_KEYS if measurement.get(key) is None]


def has_soma_branch(tree):
    return any(branch.type == "soma" for branch in tree.branches)


def compute_braincell_metrics(tree):
    soma_branches = [branch for branch in tree.branches if branch.type == "soma"]
    non_soma_branches = [branch for branch in tree.branches if branch.type != "soma"]
    return {
        "n_stems": int(tree.n_stems),
        "n_branch": int(len(non_soma_branches)),
        "n_bifs": int(sum(branch.n_children >= 2 for branch in non_soma_branches)),
        "length": float(sum(branch.length.to_decimal(u.um) for branch in non_soma_branches)),
        "surface": float(sum(branch.area.to_decimal(u.um ** 2) for branch in non_soma_branches)),
        "volume": float(sum(branch.volume.to_decimal(u.um ** 3) for branch in non_soma_branches)),
        "pathDistance": float(tree.max_path_distance_excluding_soma.to_decimal(u.um)),
        "eucDistance": float(tree.max_euclidean_distance_excluding_soma.to_decimal(u.um)),
        "soma_Surface": float(sum(branch.area.to_decimal(u.um ** 2) for branch in soma_branches)),
    }


def compute_neuromorpho_metrics(measurement):
    return {
        "n_stems": int(round(float(measurement["n_stems"]))),
        "n_branch": int(round(float(measurement["n_branch"]))),
        "n_bifs": int(round(float(measurement["n_bifs"]))),
        "length": float(measurement["length"]),
        "surface": float(measurement["surface"]),
        "volume": float(measurement["volume"]),
        "pathDistance": float(measurement["pathDistance"]),
        "eucDistance": float(measurement["eucDistance"]),
        "soma_Surface": float(measurement["soma_Surface"]),
    }


def compare_tree_with_measurement(tree, measurement):
    braincell_metrics = compute_braincell_metrics(tree)
    neuromorpho_metrics = compute_neuromorpho_metrics(measurement)
    metric_results = {}

    for key, target in neuromorpho_metrics.items():
        actual = braincell_metrics[key]
        if key in EXACT_MATCH_KEYS:
            abs_diff = abs(actual - target)
            rel_diff = None
            passed = actual == target
        else:
            abs_diff = abs(actual - target)
            if abs(target) <= 1e-12:
                rel_diff = 0.0 if abs_diff <= 1e-12 else None
                passed = abs_diff <= 1e-12
            else:
                rel_diff = abs_diff / abs(target)
                passed = rel_diff < FLOAT_THRESHOLDS[key]

        metric_results[key] = {
            "braincell": actual,
            "neuromorpho": target,
            "abs_diff": abs_diff,
            "rel_diff": rel_diff,
            "threshold": FLOAT_THRESHOLDS.get(key),
            "pass": passed,
        }

    non_soma_float_keys = [
        key for key in metric_results
        if key not in EXACT_MATCH_KEYS and key != "soma_Surface"
    ]
    non_soma_rel_diffs = [
        metric_results[key]["rel_diff"]
        for key in non_soma_float_keys
        if metric_results[key]["rel_diff"] is not None
    ]

    integer_pass = all(metric_results[key]["pass"] for key in EXACT_MATCH_KEYS)
    non_soma_float_pass = all(metric_results[key]["pass"] for key in non_soma_float_keys)
    soma_surface_pass = metric_results["soma_Surface"]["pass"]

    return {
        "braincell_metrics": braincell_metrics,
        "neuromorpho_metrics": neuromorpho_metrics,
        "metric_results": metric_results,
        "integer_pass": integer_pass,
        "non_soma_float_pass": non_soma_float_pass,
        "soma_surface_pass": soma_surface_pass,
        "overall_pass": integer_pass and non_soma_float_pass and soma_surface_pass,
        "non_soma_max_rel_diff": max(non_soma_rel_diffs) if non_soma_rel_diffs else 0.0,
        "soma_surface_rel_diff": metric_results["soma_Surface"]["rel_diff"],
    }


## 1. Batch query and cache neurons

Use `q` and `fq` as the NeuroMorpho query, then cache every matched neuron into:

`develop_doc/data/neuromorpho/<neuron_id>/`

Each neuron folder will contain:

- the standardized `SWC`, the original file, or both
- `metadata.json`
- the full NeuroMorpho measurement payload and links needed to trace the neuron later
            


In [3]:
q = "species:mouse"
fq = [
    "brain_region:cerebellum",
]
size = 100
page_start = 5
max_pages = 5
sort = "neuron_id,asc"
download_mode = "standard"  # choose from: "standard", "original", "both"
overwrite = False
output_root = repo_root / "develop_doc" / "data" / "neuromorpho"

client = NeuroMorphoClient(cache_dir=output_root)
neurons, query_urls = client.search_batch(
    q=q,
    fq=fq,
    size=size,
    page_start=page_start,
    max_pages=max_pages,
    sort=sort,
)
if not neurons:
    raise RuntimeError("No NeuroMorpho records matched the current filters. Relax q/fq and run again.")

print("q:", q)
print("fq:", fq)
print("size:", size)
print("page_start:", page_start)
print("max_pages:", max_pages)
print("download_mode:", download_mode)
print("output_root:", output_root)
print()
for query_url in query_urls:
    print("query_url:", query_url)
print()
print("matched_neurons:", len(neurons))
print()
for neuron in neurons:
    print(f"id={neuron.neuron_id} name={neuron.neuron_name}")
    print("  archive:", neuron.archive)
    print("  brain_region:", neuron.brain_region)
    print("  original_format:", neuron.original_format)
    print()


q: species:mouse
fq: ['brain_region:cerebellum']
size: 100
page_start: 5
max_pages: 5
download_mode: standard
output_root: /home/swl/braincell/develop_doc/data/neuromorpho

query_url: https://neuromorpho.org/api/neuron/select/?q=species%3Amouse&size=100&page=5&sort=neuron_id%2Casc&fq=brain_region%3Acerebellum
query_url: https://neuromorpho.org/api/neuron/select/?q=species%3Amouse&size=100&page=6&sort=neuron_id%2Casc&fq=brain_region%3Acerebellum
query_url: https://neuromorpho.org/api/neuron/select/?q=species%3Amouse&size=100&page=7&sort=neuron_id%2Casc&fq=brain_region%3Acerebellum
query_url: https://neuromorpho.org/api/neuron/select/?q=species%3Amouse&size=100&page=8&sort=neuron_id%2Casc&fq=brain_region%3Acerebellum
query_url: https://neuromorpho.org/api/neuron/select/?q=species%3Amouse&size=100&page=9&sort=neuron_id%2Casc&fq=brain_region%3Acerebellum

matched_neurons: 500

id=183577 name=Wild_14-Week_VGLUT2_6_g
  archive: Dhanya
  brain_region: ['cerebellum', 'cerebellar cortex', 'Purk

In [4]:
download_records = []
for neuron in neurons:
    record = client.download(
        neuron,
        output_dir=output_root,
        mode=download_mode,
        overwrite=overwrite,
    )
    download_records.append(record)

    print(f"cached neuron_id={neuron.neuron_id} name={neuron.neuron_name}")
    print("  folder:", record.folder)
    print("  metadata:", record.metadata_path)
    for item in record.download_items:
        print(f"  {item.kind}: {item.filename}")
        print("    url:", item.url)
        print("    path:", item.path)
        print("    downloaded_now:", item.downloaded_now)
        print("    reason:", item.reason)
    print()

if download_records:
    sample_metadata = load_cached_metadata(download_records[0].folder)
    sample_detail = client.describe(neurons[0])
    print("sample_metadata_file:", download_records[0].metadata_path)
    print("sample_measurement_url:", sample_metadata.get("measurement_url"))
    print("sample_thumbnail_url:", sample_detail.thumbnail_url)
    print("sample_standard_swc_url:", sample_detail.standard_swc_url)
    print("sample_original_file_url:", sample_detail.original_file_url)
    print("sample_measurement:")
    print(sample_metadata["measurement"])


cached neuron_id=183577 name=Wild_14-Week_VGLUT2_6_g
  folder: /home/swl/braincell/develop_doc/data/neuromorpho/183577
  metadata: /home/swl/braincell/develop_doc/data/neuromorpho/183577/metadata.json
  standard: Wild_14-Week_VGLUT2_6_g.CNG.swc
    url: https://neuromorpho.org/dableFiles/dhanya/CNG%20version/Wild_14-Week_VGLUT2_6_g.CNG.swc
    path: /home/swl/braincell/develop_doc/data/neuromorpho/183577/Wild_14-Week_VGLUT2_6_g.CNG.swc
    downloaded_now: False
    reason: None

cached neuron_id=183578 name=STIM1KO_14-week_VGLUT2_2_o
  folder: /home/swl/braincell/develop_doc/data/neuromorpho/183578
  metadata: /home/swl/braincell/develop_doc/data/neuromorpho/183578/metadata.json
  standard: STIM1KO_14-week_VGLUT2_2_o.CNG.swc
    url: https://neuromorpho.org/dableFiles/dhanya/CNG%20version/STIM1KO_14-week_VGLUT2_2_o.CNG.swc
    path: /home/swl/braincell/develop_doc/data/neuromorpho/183578/STIM1KO_14-week_VGLUT2_2_o.CNG.swc
    downloaded_now: False
    reason: None

cached neuron_id=183

## 2. Scan cached folders and batch compare

This step does not re-query NeuroMorpho. It only scans local folders under `develop_doc/data/neuromorpho/`, loads each cached standardized `SWC`, and compares it against the cached `measurement` stored in `metadata.json`.
            


In [5]:
if output_root.exists():
    cache_dirs = sorted(path for path in output_root.iterdir() if path.is_dir())
else:
    cache_dirs = []

comparison_runs = []
summary = {
    "scanned": 0,
    "compared": 0,
    "skipped": 0,
    "passed": 0,
    "failed": 0,
    "errors": 0,
}

if not cache_dirs:
    print("No cached neuron folders were found under", output_root)
else:
    for folder in cache_dirs:
        summary["scanned"] += 1
        metadata_path = folder / "metadata.json"
        if not metadata_path.exists():
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: missing metadata.json")
            continue

        try:
            metadata = load_cached_metadata(folder)
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to read metadata.json: {exc}")
            continue

        measurement = metadata.get("measurement")
        if not measurement:
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: metadata.json does not contain measurement")
            continue

        missing_keys = missing_measurement_keys(measurement)
        if missing_keys:
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: incomplete measurement for keys {missing_keys}")
            continue

        swc_path = find_standard_swc(folder, metadata)
        if swc_path is None or not swc_path.exists():
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: no standardized SWC found")
            continue

        try:
            tree, report = Morpho.from_swc(swc_path, return_report=True, mode="neuromorpho")
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to compare {swc_path}: {exc}")
            continue

        if not has_soma_branch(tree):
            summary["skipped"] += 1
            print(f"[skip] {folder.name}: no soma branches in {swc_path}")
            continue

        try:
            comparison = compare_tree_with_measurement(tree, measurement)
        except Exception as exc:
            summary["errors"] += 1
            print(f"[error] {folder.name}: failed to compare {swc_path}: {exc}")
            continue

        summary["compared"] += 1
        if comparison["overall_pass"]:
            summary["passed"] += 1
        else:
            summary["failed"] += 1

        comparison_runs.append(
            {
                "folder": str(folder),
                "neuron_id": metadata.get("neuron_id"),
                "neuron_name": metadata.get("neuron_name"),
                "swc_path": str(swc_path),
                "comparison": comparison,
            }
        )

        print("=" * 80)
        print(f"neuron_id: {metadata.get('neuron_id')}")
        print(f"neuron_name: {metadata.get('neuron_name')}")
        print(f"folder: {folder}")
        print(f"swc_path: {swc_path}")
        print()

        for key, result in comparison["metric_results"].items():
            rel_text = "n/a" if result["rel_diff"] is None else f"{result['rel_diff']:.6%}"
            print(
                f"{key}: braincell={result['braincell']:.6f}, neuromorpho={result['neuromorpho']:.6f}, "
                f"abs_diff={result['abs_diff']:.6f}, rel_diff={rel_text}, pass={result['pass']}"
            )

        print()
        print(f"integer_pass: {comparison['integer_pass']}")
        print(f"non_soma_max_rel_diff: {comparison['non_soma_max_rel_diff']:.6%}")
        print(
            "soma_surface_rel_diff:",
            "n/a" if comparison["soma_surface_rel_diff"] is None else f"{comparison['soma_surface_rel_diff']:.6%}",
        )
        print(f"non_soma_float_pass: {comparison['non_soma_float_pass']}")
        print(f"soma_surface_pass: {comparison['soma_surface_pass']}")
        print(f"overall_pass: {comparison['overall_pass']}")

    print("=" * 80)
    print("scan summary")
    print("scanned:", summary["scanned"])
    print("compared:", summary["compared"])
    print("skipped:", summary["skipped"])
    print("passed:", summary["passed"])
    print("failed:", summary["failed"])
    print("errors:", summary["errors"])

comparison_runs


neuron_id: 10069
neuron_name: Purkinje-slice-ageP43-6
folder: /home/swl/braincell/develop_doc/data/neuromorpho/10069
swc_path: /home/swl/braincell/develop_doc/data/neuromorpho/10069/Purkinje-slice-ageP43-6.CNG.swc

n_stems: braincell=1.000000, neuromorpho=1.000000, abs_diff=0.000000, rel_diff=n/a, pass=True
n_branch: braincell=757.000000, neuromorpho=757.000000, abs_diff=0.000000, rel_diff=n/a, pass=True
n_bifs: braincell=378.000000, neuromorpho=378.000000, abs_diff=0.000000, rel_diff=n/a, pass=True
length: braincell=6438.618247, neuromorpho=6438.620000, abs_diff=0.001753, rel_diff=0.000027%, pass=True
surface: braincell=26745.990234, neuromorpho=25786.900000, abs_diff=959.090234, rel_diff=3.719292%, pass=False
volume: braincell=9198.245117, neuromorpho=8802.160000, abs_diff=396.085117, rel_diff=4.499863%, pass=False
pathDistance: braincell=265.975156, neuromorpho=265.970000, abs_diff=0.005156, rel_diff=0.001939%, pass=True
eucDistance: braincell=203.025389, neuromorpho=203.020000, abs

[{'folder': '/home/swl/braincell/develop_doc/data/neuromorpho/10069',
  'neuron_id': 10069,
  'neuron_name': 'Purkinje-slice-ageP43-6',
  'swc_path': '/home/swl/braincell/develop_doc/data/neuromorpho/10069/Purkinje-slice-ageP43-6.CNG.swc',
  'comparison': {'braincell_metrics': {'n_stems': 1,
    'n_branch': 757,
    'n_bifs': 378,
    'length': 6438.618247172927,
    'surface': 26745.990234375,
    'volume': 9198.2451171875,
    'pathDistance': 265.975156418138,
    'eucDistance': 203.0253888064249,
    'soma_Surface': 1104.30126953125},
   'neuromorpho_metrics': {'n_stems': 1,
    'n_branch': 757,
    'n_bifs': 378,
    'length': 6438.62,
    'surface': 25786.9,
    'volume': 8802.16,
    'pathDistance': 265.97,
    'eucDistance': 203.02,
    'soma_Surface': 1103.58},
   'metric_results': {'n_stems': {'braincell': 1,
     'neuromorpho': 1,
     'abs_diff': 0,
     'rel_diff': None,
     'threshold': None,
     'pass': True},
    'n_branch': {'braincell': 757,
     'neuromorpho': 757,
